# **HANDS - ON: WHATSAPP CLOUD API**

Una vez vista la masterclass ***WhatsApp Cloud API***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python***  y el modelo de ***llama 3.1*** de Meta.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/19F84m5PKCMKEOHxL7z3MUqN98HEhwLWS?usp=drive_link).

## **LOGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [3]:
# Leer credenciales desde Colab Secrets
from google.colab import userdata

def leer_secret(nombre):
    try:
        valor = userdata.get(nombre)
    except Exception:
        raise RuntimeError(f"Configura el Secret {nombre} y autoriza su acceso en Colab.") from None
    if not isinstance(valor, str) or not valor.strip():
        raise RuntimeError(f"El Secret {nombre} está vacío.")
    return valor.strip()

WHATSAPP_ACCESS_TOKEN = leer_secret("WHATSAPP_ACCESS_TOKEN")
WHATSAPP_PHONE_NUMBER_ID = leer_secret("WHATSAPP_PHONE_NUMBER_ID")
WHATSAPP_VERIFY_TOKEN = leer_secret("WHATSAPP_VERIFY_TOKEN")
NGROK_AUTHTOKEN = leer_secret("NGROK_AUTHTOKEN")
# Usa la versión habilitada en tu aplicación de Meta (formato vNN.N).
WHATSAPP_API_VERSION = leer_secret("WHATSAPP_API_VERSION")
import re
if not re.fullmatch(r"v[0-9]+\.[0-9]+", WHATSAPP_API_VERSION):
    raise ValueError("WHATSAPP_API_VERSION debe tener el formato vNN.N.")
WHATSAPP_URL = (
    f"https://graph.facebook.com/{WHATSAPP_API_VERSION}/"
    f"{WHATSAPP_PHONE_NUMBER_ID}/messages"
)
# No imprimir credenciales ni cabeceras de autorización.


#### **INSTALACIÓN Y ARRANQUE DE OLLAMA**

Ollama no viene preinstalado en la máquina virtual de Google Colab, por lo que se debe instalar y arrancar en segundo plano al inicio de cada sesión.

In [4]:
!apt-get update -qq
!apt-get install -y zstd

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 135 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (372 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../z

In [5]:
# Instalar Ollama y las dependencias del ejercicio en Google Colab.
# Selecciona un entorno con GPU antes de ejecutar para usar Llama 3.1 8B.
import shutil
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "requests"],
    check=True,
)
if shutil.which("ollama") is None:
    subprocess.run(
        ["bash", "-o", "pipefail", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=True,
    )


In [6]:
# Iniciar el servidor de Ollama; reutilizarlo si ya está disponible.
import os
import time
import requests

OLLAMA_URL = "http://127.0.0.1:11434"
MODELO = "llama3.1:8b"
try:
    requests.get(f"{OLLAMA_URL}/api/tags", timeout=2).raise_for_status()
except requests.RequestException:
    entorno_ollama = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434")
    proceso_ollama = subprocess.Popen(
        ["ollama", "serve"], env=entorno_ollama,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    for intento in range(60):
        try:
            requests.get(f"{OLLAMA_URL}/api/tags", timeout=2).raise_for_status()
            break
        except requests.RequestException:
            if proceso_ollama.poll() is not None:
                raise RuntimeError("Ollama terminó antes de estar disponible.") from None
            time.sleep(1)
    else:
        raise RuntimeError("Ollama no respondió en el tiempo esperado.")
subprocess.run(
    ["ollama", "pull", MODELO],
    env=dict(os.environ, OLLAMA_HOST="127.0.0.1:11434"), check=True,
)


CompletedProcess(args=['ollama', 'pull', 'llama3.1:8b'], returncode=0)

### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

Con Ollama corriendo, se arma la lógica del asistente: la estructura del servidor con sus registros en memoria, la normalización de números mexicanos (necesaria para que la API de envío acepte el número que entrega el webhook), y la función que genera la respuesta real con Llama 3.1.

In [7]:
# Configurar el servidor del webhook y los registros de esta práctica.
import threading
import hmac
import json
from fastapi import FastAPI, BackgroundTasks, Request
from fastapi.responses import PlainTextResponse, JSONResponse

# Detener el servidor anterior ANTES de redefinir funciones o registros.
# Al pasar al Challenge se inicia una práctica nueva, con usuarios nuevos.
if "hilo_servidor" in globals() and hilo_servidor.is_alive():
    servidor.should_exit = True
    hilo_servidor.join(timeout=180)
    if hilo_servidor.is_alive():
        raise RuntimeError("Espera a que termine el servidor anterior antes de continuar.")

app = FastAPI()
mensajes_recibidos = []  # Payloads en memoria: pueden contener datos personales.
usuarios_conocidos = set()
mensajes_en_proceso = set()
mensajes_procesados = set()
bloqueos_usuarios = {}
lock_registros = threading.Lock()


In [8]:
# Definir función de normalización de número.
def normalizar_numero_mx(numero):
    if not isinstance(numero, str):
        return ""
    numero = numero.strip().removeprefix("+")
    if not numero.isascii() or not numero.isdigit():
        return ""
    # WhatsApp puede entregar 521 + 10 dígitos; enviar 52 + 10 dígitos.
    if numero.startswith("521") and len(numero) == 13:
        numero = "52" + numero[3:]
    return numero if 8 <= len(numero) <= 15 else ""


In [9]:
# Definir función de generación de respuesta con Llama 3.1.
import textwrap

PROMPT_ASISTENTE = (
    "Eres un asistente de estatus de pedidos. Responde en español y con amabilidad. "
    "Orienta solamente sobre consultas de pedidos. No tienes acceso a un sistema "
    "de pedidos ni datos verificados: no inventes estados, fechas, números, enlaces "
    "o acciones realizadas. Explica esa limitación si te piden un estatus real. "
    "No solicites contraseñas ni datos de pago. "
)

def generar_respuesta(texto):
    try:
        resultado = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={"model": MODELO, "system": PROMPT_ASISTENTE,
                  "prompt": texto, "stream": False,
                  "options": {"temperature": 0.2, "num_predict": 120}},
            timeout=(5, 120),
        )
        resultado.raise_for_status()
        datos = resultado.json()
        respuesta = datos.get("response") if isinstance(datos, dict) else None
        if not isinstance(respuesta, str) or not respuesta.strip():
            raise ValueError("Respuesta vacía del modelo")
        return respuesta.strip()
    except (requests.RequestException, ValueError):
        print("No se pudo generar la respuesta con Ollama; revisa el servicio y el modelo.")
        return "Lo siento, no pude procesar tu consulta ahora. Inténtalo de nuevo en un momento."


## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante.

**Aclaración vigente:** La recepción delega el procesamiento a `BackgroundTasks` para devolver rápidamente el ACK HTTP a Meta; la generación con Llama (en el Hands-On) y el envío continúan posteriormente. Esto no garantiza ausencia de reintentos, duplicados ni entrega de la respuesta. La deduplicación actual se guarda en memoria; reiniciar la sesión o volver a configurar la aplicación elimina ese estado.

**Explicación original conservada — la afirmación sobre evitar reintentos y duplicados queda matizada por la aclaración anterior:**

> La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta, evitando así que Meta reintente la entrega y se dupliquen las respuestas.


In [10]:
# Definir endpoint de verificación de credenciales.
@app.get("/webhook")
async def verificar_webhook(request: Request):
    parametros = request.query_params
    token = parametros.get("hub.verify_token", "")
    if (parametros.get("hub.mode") == "subscribe"
            and hmac.compare_digest(token.encode(), WHATSAPP_VERIFY_TOKEN.encode())
            and parametros.get("hub.challenge")):
        return PlainTextResponse(parametros["hub.challenge"])
    return PlainTextResponse("Verificación rechazada", status_code=403)


In [11]:
# Definir endpoint de recepción de mensajes.
def extraer_mensajes_texto(payload):
    if not isinstance(payload, dict) or payload.get("object") != "whatsapp_business_account":
        return
    entradas = payload.get("entry")
    for entrada in entradas if isinstance(entradas, list) else []:
        cambios = entrada.get("changes") if isinstance(entrada, dict) else None
        for cambio in cambios if isinstance(cambios, list) else []:
            valor = cambio.get("value") if isinstance(cambio, dict) else None
            if not isinstance(valor, dict):
                continue
            metadata = valor.get("metadata")
            if not isinstance(metadata, dict) or metadata.get("phone_number_id") != WHATSAPP_PHONE_NUMBER_ID:
                continue
            mensajes = valor.get("messages")
            for mensaje in mensajes if isinstance(mensajes, list) else []:
                if not isinstance(mensaje, dict) or mensaje.get("type") != "text":
                    continue
                texto = mensaje.get("text")
                cuerpo = texto.get("body") if isinstance(texto, dict) else None
                numero = normalizar_numero_mx(mensaje.get("from"))
                identificador = mensaje.get("id")
                if numero and isinstance(cuerpo, str) and cuerpo.strip() and isinstance(identificador, str) and identificador:
                    yield identificador, numero, cuerpo.strip()

@app.post("/webhook")
async def recibir_webhook(request: Request, background_tasks: BackgroundTasks):
    try:
        payload = await request.json()
    except (ValueError, UnicodeDecodeError):
        return JSONResponse({"error": "JSON inválido"}, status_code=400)
    mensajes_recibidos.append(payload)
    for identificador, numero, texto in extraer_mensajes_texto(payload):
        with lock_registros:
            if identificador in mensajes_en_proceso or identificador in mensajes_procesados:
                continue
            mensajes_en_proceso.add(identificador)
            bloqueos_usuarios.setdefault(numero, threading.Lock())
        background_tasks.add_task(procesar_respuesta, numero, texto, identificador)
    # Estados de entrega/lectura, imágenes, audio y payloads sin texto se ignoran.
    return {"status": "ok"}


In [12]:
# Definir función de envío de respuesta mediante WhatsApp Cloud API.
def enviar_respuesta(numero, texto):
    try:
        resultado = requests.post(
            WHATSAPP_URL,
            headers={"Authorization": f"Bearer {WHATSAPP_ACCESS_TOKEN}"},
            json={"messaging_product": "whatsapp", "recipient_type": "individual",
                  "to": normalizar_numero_mx(numero), "type": "text",
                  "text": {"preview_url": False, "body": texto}},
            timeout=(5, 30),
        )
        if not resultado.ok:
            # No imprimir el cuerpo ni la URL: pueden contener datos sensibles.
            print(f"WhatsApp rechazó el envío (HTTP {resultado.status_code}). Revisa la configuración en Meta.")
            return False
        return True
    except requests.RequestException:
        print("No se pudo contactar con WhatsApp para enviar la respuesta.")
        return False


In [13]:
# Definir función de procesamiento de respuesta.
SALUDO_INICIAL = (
    "¡Hola! Soy tu asistente de estatus de pedidos.\n"
    "Cuéntame tu consulta; puedo orientarte, aunque no tengo acceso al estatus real de tu pedido."
)

def procesar_respuesta(numero, texto, identificador):
    # Función síncrona: BackgroundTasks la ejecuta fuera del bucle asíncrono.
    # El bloqueo por número evita dos saludos simultáneos al mismo usuario.
    try:
        with bloqueos_usuarios[numero]:
            nuevo = numero not in usuarios_conocidos
            respuesta = SALUDO_INICIAL if nuevo else generar_respuesta(texto)
            if enviar_respuesta(numero, respuesta):
                with lock_registros:
                    usuarios_conocidos.add(numero)
                    mensajes_procesados.add(identificador)
            # Si falla el envío, un primer contacto sigue pendiente de saludo.
    except Exception:
        print("No se pudo completar el procesamiento del mensaje; revisa los servicios.")
    finally:
        with lock_registros:
            mensajes_en_proceso.discard(identificador)


### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers.

**Cambio de túnel:** Cada vez que inicies o reinicies ngrok, comprueba la URL pública de esa ejecución. Si cambió, actualiza el callback de Meta a `<URL_NGROK>/webhook`, manteniendo el mismo valor del Secret `WHATSAPP_VERIFY_TOKEN` correspondiente a esta aplicación. El código imprime el callback completo: no añadas `/webhook` dos veces. Una URL almacenada en un output anterior no acredita que ese túnel siga activo.


\begin{equation} WhatsApp  \, \, \, \Rightarrow \, \, \, Configuración  \, \,de  \, \,la  \, \,API  \, \, \, \Rightarrow \, \, \, Configurar  \, \,webhooks \end{equation}

In [14]:
# Instalar e iniciar túnel público con ngrok.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
import uvicorn
from pyngrok import ngrok
from pyngrok.conf import PyngrokConfig

if "hilo_servidor" in globals() and hilo_servidor.is_alive():
    servidor.should_exit = True
    hilo_servidor.join(timeout=180)
    if hilo_servidor.is_alive():
        raise RuntimeError("El servidor anterior aún está procesando mensajes.")
if "tunel" in globals():
    ngrok.disconnect(tunel.public_url)

# Desactivar access_log evita registrar el verify token de la URL de verificación.
servidor = uvicorn.Server(uvicorn.Config(
    app, host="0.0.0.0", port=8000, log_level="warning", access_log=False,
))
hilo_servidor = threading.Thread(target=servidor.run, daemon=True)
hilo_servidor.start()
for intento in range(100):
    if servidor.started:
        break
    if not hilo_servidor.is_alive():
        raise RuntimeError("Uvicorn no pudo arrancar; revisa el puerto 8000.")
    time.sleep(0.1)
else:
    servidor.should_exit = True
    raise RuntimeError("Uvicorn no inició en el tiempo esperado.")
try:
    tunel = ngrok.connect(
        addr=8000, proto="http", bind_tls=True,
        pyngrok_config=PyngrokConfig(auth_token=NGROK_AUTHTOKEN),
    )
except Exception:
    servidor.should_exit = True
    hilo_servidor.join(timeout=10)
    raise RuntimeError("No se pudo abrir ngrok; revisa tu cuenta y su Secret.") from None
print("Callback URL para Meta:", tunel.public_url + "/webhook")
print("Usa el valor de WHATSAPP_VERIFY_TOKEN en Meta y suscribe el campo messages.")
# La memoria se pierde al reiniciar Colab o volver a configurar la aplicación.
# Mantén esta sesión activa durante las pruebas; el ACK no garantiza el envío.


Callback URL para Meta: https://0a13-34-143-154-195.ngrok-free.app/webhook
Usa el valor de WHATSAPP_VERIFY_TOKEN en Meta y suscribe el campo messages.


In [18]:
print("Payloads recibidos:", len(mensajes_recibidos))
print(mensajes_recibidos[-1] if mensajes_recibidos else "Ninguno")

Payloads recibidos: 1
{'object': 'whatsapp_business_account', 'entry': [{'id': '0', 'changes': [{'field': 'messages', 'value': {'messaging_product': 'whatsapp', 'metadata': {'display_phone_number': '16505551111', 'phone_number_id': '123456123'}, 'contacts': [{'profile': {'name': 'test user name'}, 'wa_id': '16315551181', 'user_id': 'US.13491208655302741918'}], 'messages': [{'id': 'ABGGFlA5Fpa', 'timestamp': '1504902988', 'from': '16315551181', 'from_user_id': 'US.13491208655302741918', 'type': 'text', 'text': {'body': 'this is a text message'}}]}}]}]}


### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje para confirmar que el saludo inicial funciona, y un segundo mensaje para confirmar que ya se recibe una respuesta generada por Llama.

**Pruebas reales pendientes del Hands-On:** Ejecuta esta sección en orden, desde una sesión nueva con GPU y los Secrets autorizados, hasta exponer el servidor. Configura el callback vigente en Meta y comprueba la suscripción a `messages`. Desde WhatsApp envía `Hola` para comprobar el saludo y, después, una consulta sobre pedidos para comprobar una respuesta de Llama. El modelo debe reconocer que no dispone de estatus reales.

Los mensajes impresos por las dos celdas siguientes son recordatorios, no resultados de pruebas. La descarga del modelo o la impresión de una URL tampoco demuestra una respuesta recibida. Tras efectuar las pruebas, registra únicamente observaciones reales, con fecha y sección probada, y guarda el notebook; no incluyas tokens, números personales ni payloads completos.

**Estado de la evidencia:** Se conservan outputs de ejecuciones anteriores, sin presentarlos como validación de este candidato final. Antes de entregar, reinicia la sesión, limpia las salidas antiguas y ejecuta las celdas en orden para generar outputs nuevos de forma auténtica. Verifica las respuestas en WhatsApp: el envío en segundo plano no imprime necesariamente el texto recibido en el notebook.


In [15]:
# Mandar mensaje al bot para recibir un saludo: prueba MANUAL desde WhatsApp.
# Con el webhook verificado y messages suscrito, envía «Hola» al número de Meta.
# Si Meta usa un número de prueba, autoriza antes tu número como destinatario.
# Debes recibir SALUDO_INICIAL; ese primer texto no se consulta con Llama.
# Usa un usuario nuevo o reinicia la sección para repetir la prueba del saludo.
print("Prueba pendiente: envía Hola desde WhatsApp y comprueba el saludo inicial.")


Prueba pendiente: envía Hola desde WhatsApp y comprueba el saludo inicial.


In [16]:
# Seguir conversando con el bot para interactuar con Llama: prueba MANUAL.
# Desde el mismo número y después del saludo, envía:
# «¿Puedes consultar el estatus real de mi pedido?»
# Debe responder Llama, reconocer que no tiene acceso a pedidos y no inventar datos.
# Prueba también una imagen o audio: se ignoran y no deben causar errores HTTP.
# No imprimas mensajes_recibidos al compartir el notebook: contiene datos personales.
print("Prueba pendiente: envía una segunda consulta desde el mismo número.")


Prueba pendiente: envía una segunda consulta desde el mismo número.


## **CHALLENGE: ASISTENTE DE PEDIDOS**

> **Aclaración de la consigna vigente:** Este Challenge se resuelve con un diccionario de pedidos de ejemplo y reglas fijas, sin modelos. Las referencias originales a Ollama/Llama que aparecen abajo se conservan como texto del profesor, pero quedan sustituidas para esta entrega por consulta determinista de pedidos, validación de entrada y respuestas fijas. Puede ejecutarse desde esta sección en una sesión nueva, sin ejecutar el Hands-On ni usar GPU.

### Challenge: Asistente de Estatus de Pedidos

**Consigna vigente y alcance:** Ollama/Llama corresponde legítimamente al Hands-On. Este Challenge consulta el diccionario `pedidos` y aplica reglas fijas; no usa ningún modelo para determinar ni redactar el estatus. Los textos originales sobre modelos se identifican abajo como material sustituido y no son pasos que deban ejecutarse aquí.

**Orden de ejecución:** En una sesión nueva, ejecuta las celdas de código 39 a 56 en orden; no necesitas ejecutar el Hands-On ni seleccionar GPU para el Challenge. La entrega sí debe conservar el Hands-On completo. Si vienes de probar el Hands-On, la celda 39 detiene su servidor; el Challenge reinicia los registros en memoria y la celda 54 vuelve a exponer el servidor. Comprueba la URL nueva antes de enviar mensajes. No mezcles celdas aisladas de ambas secciones con el servidor activo.

**Usuarios:** Un usuario nuevo recibe saludo **y** resultado de su primera consulta; por ejemplo, `1001` devuelve el saludo y `En preparación`. Una vez aceptado el envío, queda registrado. Los mensajes posteriores reciben únicamente el resultado correspondiente. Las entradas inválidas o pedidos inexistentes también reciben su respuesta fija, con saludo si es el primer contacto.

**Evidencia pendiente:** Los contadores y outputs anteriores no prueban que esta versión del Challenge haya sido ejecutada. Las celdas 55 y 56 describen pruebas manuales pendientes, no resultados. Ejecuta y guarda esta versión en Colab antes de entregar; comprueba que la copia compartida contiene estas reglas y no código de modelos dentro del Challenge.


Una vez visto el ***Hands-On: WhatsApp Cloud API***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

A partir del asistente de pedidos construido, se agregan dos mejoras:

* Un mensaje de bienvenida para usuarios nuevos.
* Un tono de respuesta más controlado.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**Consigna vigente para este paso del Challenge:**

Instala las dependencias del webhook con la celda 39, carga el diccionario de ejemplo y los imports con la celda 40, y lee las credenciales desde Colab Secrets con la celda 41. No instales ni arranques Ollama y no descargues Llama para esta sección. Los cinco Secrets requeridos son `WHATSAPP_ACCESS_TOKEN`, `WHATSAPP_PHONE_NUMBER_ID`, `WHATSAPP_VERIFY_TOKEN`, `NGROK_AUTHTOKEN` y `WHATSAPP_API_VERSION`; autoriza el acceso del notebook sin imprimir sus valores.

**Material original conservado — sustituido por la consigna vigente anterior; no ejecutar sus indicaciones sobre modelos en el Challenge:**

> **1. Prepara el entorno:** Instala y arranca Ollama en Colab (no viene preinstalado en la máquina virtual, así que se instala y se arranca en segundo plano al inicio de la sesión), descarga el modelo Llama 3.1 8B, y carga las credenciales de WhatsApp desde **Colab Secrets**.


In [19]:
# Detener el servidor anterior ANTES de redefinir funciones o registros.
# Al pasar al Challenge se inicia una práctica nueva, con usuarios nuevos.
if "hilo_servidor" in globals() and hilo_servidor.is_alive():
    servidor.should_exit = True
    hilo_servidor.join(timeout=180)
    if hilo_servidor.is_alive():
        raise RuntimeError("Espera a que termine el servidor anterior antes de continuar.")

# Instalar únicamente las dependencias del webhook del Challenge.
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "requests"],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn', 'requests'], returncode=0)

In [20]:
# Datos académicos de ejemplo: no representan pedidos reales ni un catálogo oficial.
import time
import requests

pedidos = {
    "1001": "En preparación",
    "1002": "Enviado",
    "1003": "Entregado",
}


In [21]:
# Leer credenciales desde Colab Secrets
from google.colab import userdata

def leer_secret(nombre):
    try:
        valor = userdata.get(nombre)
    except Exception:
        raise RuntimeError(f"Configura el Secret {nombre} y autoriza su acceso en Colab.") from None
    if not isinstance(valor, str) or not valor.strip():
        raise RuntimeError(f"El Secret {nombre} está vacío.")
    return valor.strip()

WHATSAPP_ACCESS_TOKEN = leer_secret("WHATSAPP_ACCESS_TOKEN")
WHATSAPP_PHONE_NUMBER_ID = leer_secret("WHATSAPP_PHONE_NUMBER_ID")
WHATSAPP_VERIFY_TOKEN = leer_secret("WHATSAPP_VERIFY_TOKEN")
NGROK_AUTHTOKEN = leer_secret("NGROK_AUTHTOKEN")
# Usa la versión habilitada en tu aplicación de Meta (formato vNN.N).
WHATSAPP_API_VERSION = leer_secret("WHATSAPP_API_VERSION")
import re
if not re.fullmatch(r"v[0-9]+\.[0-9]+", WHATSAPP_API_VERSION):
    raise ValueError("WHATSAPP_API_VERSION debe tener el formato vNN.N.")
WHATSAPP_URL = (
    f"https://graph.facebook.com/{WHATSAPP_API_VERSION}/"
    f"{WHATSAPP_PHONE_NUMBER_ID}/messages"
)
# No imprimir credenciales ni cabeceras de autorización.


**2. Configura el servidor y el registro de usuarios conocidos:**

   * Crea el servidor FastAPI con las estructuras en memoria necesarias: `mensajes_recibidos` para la bitácora de payloads, y `usuarios_conocidos` (un `set`) para llevar registro de quién ya recibió el saludo.
   * Define la función `normalizar_numero_mx` para que los números mexicanos se envíen en el formato correcto.

In [23]:
# Configurar el servidor del webhook y los registros de esta práctica.
import threading
import hmac
import json
from fastapi import FastAPI, BackgroundTasks, Request
from fastapi.responses import PlainTextResponse, JSONResponse

# Detener el servidor anterior ANTES de redefinir funciones o registros.
# Al pasar al Challenge se inicia una práctica nueva, con usuarios nuevos.
if "hilo_servidor" in globals() and hilo_servidor.is_alive():
    servidor.should_exit = True
    hilo_servidor.join(timeout=180)
    if hilo_servidor.is_alive():
        raise RuntimeError("Espera a que termine el servidor anterior antes de continuar.")

app = FastAPI()
mensajes_recibidos = []  # Payloads en memoria: pueden contener datos personales.
usuarios_conocidos = set()
mensajes_en_proceso = set()
mensajes_procesados = set()
bloqueos_usuarios = {}
lock_registros = threading.Lock()


In [24]:
# Definir función de normalización de número.
def normalizar_numero_mx(numero):
    if not isinstance(numero, str):
        return ""
    numero = numero.strip().removeprefix("+")
    if not numero.isascii() or not numero.isdigit():
        return ""
    # WhatsApp puede entregar 521 + 10 dígitos; enviar 52 + 10 dígitos.
    if numero.startswith("521") and len(numero) == 13:
        numero = "52" + numero[3:]
    return numero if 8 <= len(numero) <= 15 else ""


**Consigna vigente para este paso del Challenge:**

La función `generar_respuesta` consulta exclusivamente el diccionario `pedidos` y devuelve textos fijos y breves. El nombre de la función no implica uso de IA. Se eliminan espacios exteriores y solo se admiten dígitos ASCII. Los ceros iniciales se conservan: `01001` no equivale a `1001`.

| Entrada | Resultado esperado; aún debe comprobarse en WhatsApp |
|---|---|
| `1001` | `En preparación` |
| `1002` | `Enviado` |
| `1003` | `Entregado` |
| `9999` o `01001` | `Pedido no encontrado` |
| `Hola`, `pedido 1001`, `-1001`, `1001.0` o dígitos no ASCII | `Entrada inválida` |
| ` 1001 ` | `En preparación` |

Son datos académicos de ejemplo. No hay prompts, generación libre ni consulta a modelos en este paso.

**Material original conservado — sustituido por la consigna vigente anterior; no ejecutar sus indicaciones sobre modelos en el Challenge:**

> **3. Genera la respuesta con un tono controlado:** Ajusta el prompt que se envía a Llama para que las respuestas se mantengan breves (máximo 2-3 líneas) y con un tono amable.


In [25]:
# Consultar el diccionario mediante reglas fijas, sin servicios de generación.
# Formato: solo dígitos ASCII; se permiten espacios al inicio/final.
# Se conservan ceros iniciales: "01001" no es el pedido "1001".
def generar_respuesta(texto):
    if not isinstance(texto, str):
        return "Entrada inválida. Envía solo el número de pedido, por ejemplo: 1001."
    numero_pedido = texto.strip()
    if not numero_pedido or not numero_pedido.isascii() or not numero_pedido.isdigit():
        return "Entrada inválida. Envía solo el número de pedido, por ejemplo: 1001."
    if numero_pedido not in pedidos:
        return "Pedido no encontrado en el catálogo de ejemplo. Verifica el número e inténtalo de nuevo."
    return f"Pedido {numero_pedido}: {pedidos[numero_pedido]} (dato de ejemplo)."


**Consigna vigente para este paso del Challenge:**

Conserva `GET /webhook` para la verificación y `POST /webhook` para la recepción. El POST valida el evento y programa con `BackgroundTasks` la consulta por reglas y el envío a WhatsApp; devuelve rápidamente el ACK mientras el procesamiento continúa posteriormente. No se espera a ningún modelo.

`BackgroundTasks` no garantiza que no haya reintentos ni duplicados. La deduplicación implementada es en memoria y se pierde al reiniciar la sesión o reconfigurar la aplicación. El ACK tampoco confirma que la respuesta haya sido entregada al teléfono.

**Material original conservado — sustituido por la consigna vigente anterior; no ejecutar sus indicaciones sobre modelos en el Challenge:**

> **4. Define los endpoints del webhook:**
>
>    * Crea el endpoint de verificación (`GET /webhook`).
>    * Crea el endpoint de recepción de mensajes (`POST /webhook`), delegando el trabajo pesado a `BackgroundTasks` para responder de inmediato a Meta sin esperar a Ollama.


In [26]:
# Definir endpoint de verificación de credenciales.
@app.get("/webhook")
async def verificar_webhook(request: Request):
    parametros = request.query_params
    token = parametros.get("hub.verify_token", "")
    if (parametros.get("hub.mode") == "subscribe"
            and hmac.compare_digest(token.encode(), WHATSAPP_VERIFY_TOKEN.encode())
            and parametros.get("hub.challenge")):
        return PlainTextResponse(parametros["hub.challenge"])
    return PlainTextResponse("Verificación rechazada", status_code=403)


In [27]:
# Definir endpoint de recepción de mensajes.
def extraer_mensajes_texto(payload):
    if not isinstance(payload, dict) or payload.get("object") != "whatsapp_business_account":
        return
    entradas = payload.get("entry")
    for entrada in entradas if isinstance(entradas, list) else []:
        cambios = entrada.get("changes") if isinstance(entrada, dict) else None
        for cambio in cambios if isinstance(cambios, list) else []:
            valor = cambio.get("value") if isinstance(cambio, dict) else None
            if not isinstance(valor, dict):
                continue
            metadata = valor.get("metadata")
            if not isinstance(metadata, dict) or metadata.get("phone_number_id") != WHATSAPP_PHONE_NUMBER_ID:
                continue
            mensajes = valor.get("messages")
            for mensaje in mensajes if isinstance(mensajes, list) else []:
                if not isinstance(mensaje, dict) or mensaje.get("type") != "text":
                    continue
                texto = mensaje.get("text")
                cuerpo = texto.get("body") if isinstance(texto, dict) else None
                numero = normalizar_numero_mx(mensaje.get("from"))
                identificador = mensaje.get("id")
                if numero and isinstance(cuerpo, str) and isinstance(identificador, str) and identificador:
                    yield identificador, numero, cuerpo.strip()

@app.post("/webhook")
async def recibir_webhook(request: Request, background_tasks: BackgroundTasks):
    try:
        payload = await request.json()
    except (ValueError, UnicodeDecodeError):
        return JSONResponse({"error": "JSON inválido"}, status_code=400)
    mensajes_recibidos.append(payload)
    for identificador, numero, texto in extraer_mensajes_texto(payload):
        with lock_registros:
            if identificador in mensajes_en_proceso or identificador in mensajes_procesados:
                continue
            mensajes_en_proceso.add(identificador)
            bloqueos_usuarios.setdefault(numero, threading.Lock())
        background_tasks.add_task(procesar_respuesta, numero, texto, identificador)
    # Estados de entrega/lectura, imágenes, audio y payloads sin texto se ignoran.
    return {"status": "ok"}


**Consigna vigente para este paso del Challenge:**

Envía la respuesta por WhatsApp Cloud API con las credenciales de Secrets. Para un número nuevo, procesa el primer mensaje con las reglas y envía saludo **y** resultado en una misma respuesta. Si ese mensaje es `1001`, incluye `En preparación`. Registra al usuario cuando el envío sea aceptado por la API. Los mensajes posteriores devuelven únicamente el resultado de la consulta, sin repetir saludo. El Challenge no usa Llama ni otro modelo.

**Material original conservado — sustituido por la consigna vigente anterior; no ejecutar sus indicaciones sobre modelos en el Challenge:**

> **5. Agrega el saludo inicial y el envío de la respuesta:**
>
>    * Define la función que envía el mensaje de WhatsApp.
>    * Define la función que decide si se manda el saludo fijo o la respuesta generada por Llama, según si el número ya está en `usuarios_conocidos`.


In [28]:
# Definir función de envío de respuesta mediante WhatsApp Cloud API.
def enviar_respuesta(numero, texto):
    try:
        resultado = requests.post(
            WHATSAPP_URL,
            headers={"Authorization": f"Bearer {WHATSAPP_ACCESS_TOKEN}"},
            json={"messaging_product": "whatsapp", "recipient_type": "individual",
                  "to": normalizar_numero_mx(numero), "type": "text",
                  "text": {"preview_url": False, "body": texto}},
            timeout=(5, 30),
        )
        if not resultado.ok:
            # No imprimir el cuerpo ni la URL: pueden contener datos sensibles.
            print(f"WhatsApp rechazó el envío (HTTP {resultado.status_code}). Revisa la configuración en Meta.")
            return False
        return True
    except requests.RequestException:
        print("No se pudo contactar con WhatsApp para enviar la respuesta.")
        return False


In [29]:
# Definir función de procesamiento de respuesta.
SALUDO_INICIAL = "¡Hola! Soy tu asistente de pedidos de ejemplo."

def procesar_respuesta(numero, texto, identificador):
    # Función síncrona: BackgroundTasks la ejecuta fuera del bucle asíncrono.
    # El bloqueo por número evita dos saludos simultáneos al mismo usuario.
    try:
        with bloqueos_usuarios[numero]:
            nuevo = numero not in usuarios_conocidos
            respuesta = generar_respuesta(texto)
            if nuevo:
                # También consultar el número recibido en el primer mensaje.
                respuesta = SALUDO_INICIAL + "\n" + respuesta
            if enviar_respuesta(numero, respuesta):
                with lock_registros:
                    usuarios_conocidos.add(numero)
                    mensajes_procesados.add(identificador)
            # Si falla el envío, un primer contacto sigue pendiente de saludo.
    except Exception:
        print("No se pudo completar el procesamiento del mensaje; revisa los servicios.")
    finally:
        with lock_registros:
            mensajes_en_proceso.discard(identificador)


**Consigna vigente para este paso del Challenge:**

Ejecuta la celda 54 para instalar `pyngrok`, iniciar Uvicorn en un hilo y abrir el túnel. En cada inicio o reinicio, comprueba la URL pública y, si cambió, actualiza Meta a `<URL_NGROK>/webhook`, manteniendo el mismo `WHATSAPP_VERIFY_TOKEN` correspondiente. La URL impresa ya incluye `/webhook`. Comprueba que `messages` esté suscrito.

**Pruebas reales pendientes — no son resultados de ejecución:**

1. Desde un usuario nuevo, envía `1001`: espera saludo y `En preparación` en una misma respuesta.
2. Desde ese mismo número, envía `1002` y `1003`: espera `Enviado` y `Entregado`, sin otro saludo.
3. Envía `9999`: espera `Pedido no encontrado`. Envía `Hola`, `pedido 1001`, `-1001` y `1001.0`: espera `Entrada inválida`.
4. Envía ` 1001 ` y `01001`: espera, respectivamente, `En preparación` y `Pedido no encontrado`. Para comprobar ASCII, envía `１００１`: espera `Entrada inválida`.
5. Repite `1002` como un mensaje nuevo: debe devolver el mismo estatus. Envía imagen o audio: se ignoran y no deben provocar errores del webhook.
6. En una sesión nueva, ejecuta solo el Challenge desde la celda 39 para comprobar su independencia del Hands-On. Revisa otra vez el callback si cambia el túnel.

Los outputs existentes son históricos; no acreditan estos casos ni la entrega de respuestas. Las celdas 55 y 56 solo imprimen recordatorios. Registra resultados observados únicamente después de probarlos, con fecha y sin datos sensibles. Que Meta acepte el callback o que FastAPI reciba un evento no demuestra que se haya recibido la respuesta en WhatsApp.

**Antes de entregar:** completa las pruebas del Hands-On y del Challenge, guarda las salidas auténticas de las ejecuciones y comprueba que no contengan credenciales ni datos personales. Guarda el `.ipynb` y la misma versión en Google Colab. Comparte esa copia como lector y verifica su acceso desde otra cuenta o una ventana sin sesión; confirma que contiene el diccionario y estas instrucciones. No presentes casos pendientes como aprobados.

**Material original conservado — sustituido por la consigna vigente anterior; no ejecutar sus indicaciones sobre modelos en el Challenge:**

> **6. Expón el servidor y prueba el flujo completo:**
>
>    * Instala `pyngrok`, abre el túnel público y arranca el servidor en un hilo aparte.
>    * Manda un primer mensaje de prueba y confirma que se recibe el saludo; manda un segundo mensaje y confirma que ya llega la respuesta generada por Llama.


In [30]:
# Instalar e iniciar túnel público con ngrok.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
import uvicorn
from pyngrok import ngrok
from pyngrok.conf import PyngrokConfig

if "hilo_servidor" in globals() and hilo_servidor.is_alive():
    servidor.should_exit = True
    hilo_servidor.join(timeout=180)
    if hilo_servidor.is_alive():
        raise RuntimeError("El servidor anterior aún está procesando mensajes.")
if "tunel" in globals():
    ngrok.disconnect(tunel.public_url)

# Desactivar access_log evita registrar el verify token de la URL de verificación.
servidor = uvicorn.Server(uvicorn.Config(
    app, host="0.0.0.0", port=8000, log_level="warning", access_log=False,
))
hilo_servidor = threading.Thread(target=servidor.run, daemon=True)
hilo_servidor.start()
for intento in range(100):
    if servidor.started:
        break
    if not hilo_servidor.is_alive():
        raise RuntimeError("Uvicorn no pudo arrancar; revisa el puerto 8000.")
    time.sleep(0.1)
else:
    servidor.should_exit = True
    raise RuntimeError("Uvicorn no inició en el tiempo esperado.")
try:
    tunel = ngrok.connect(
        addr=8000, proto="http", bind_tls=True,
        pyngrok_config=PyngrokConfig(auth_token=NGROK_AUTHTOKEN),
    )
except Exception:
    servidor.should_exit = True
    hilo_servidor.join(timeout=10)
    raise RuntimeError("No se pudo abrir ngrok; revisa tu cuenta y su Secret.") from None
print("Callback URL para Meta:", tunel.public_url + "/webhook")
print("Usa el valor de WHATSAPP_VERIFY_TOKEN en Meta y suscribe el campo messages.")
# La memoria se pierde al reiniciar Colab o volver a configurar la aplicación.
# Mantén esta sesión activa durante las pruebas; el ACK no garantiza el envío.


Callback URL para Meta: https://6930-34-143-154-195.ngrok-free.app/webhook
Usa el valor de WHATSAPP_VERIFY_TOKEN en Meta y suscribe el campo messages.


In [ ]:
# Prueba MANUAL desde WhatsApp con el webhook configurado en Meta.
# Para un usuario nuevo, envía "1001": debe recibir el saludo y
# "Pedido 1001: En preparación (dato de ejemplo)." en la misma respuesta.
# El usuario queda registrado después de un envío aceptado por la API.
# En otra consulta desde ese número no debe repetirse el saludo.
print("Prueba pendiente: envía 1001 desde un usuario nuevo y comprueba saludo y estatus.")


In [ ]:
# Pruebas MANUALES desde el mismo número:
# - "1002": Enviado; "1003": Entregado. Sin repetir saludo.
# - "9999": pedido no encontrado en el catálogo de ejemplo.
# - "Hola", "pedido 1001", "-1001", "1001.0": entrada inválida.
# - " 1001 ": En preparación; "01001": pedido no encontrado.
# Repite "1002" como un mensaje nuevo: debe devolver el mismo estatus.
# Envía imagen/audio: se ignoran sin errores HTTP.
# En una sesión nueva, ejecuta solo el Challenge para comprobar su independencia.
# No imprimas mensajes_recibidos: contiene datos personales.
print("Pruebas pendientes: pedidos existentes, inexistentes y entradas inválidas.")


In [31]:
# DIAGNÓSTICO LOCAL SIMULADO — no constituye evidencia E2E de WhatsApp.
# Ejecuta antes las definiciones del Challenge hasta procesar_respuesta.
# No hace falta ejecutar la celda de ngrok. Requiere fastapi, uvicorn y requests.
# Reutiliza el código del Challenge en una instancia aislada: datos ficticios,
# registros propios y envío capturado en memoria. Nunca envía estos casos a Meta.
# Si hay un servidor del notebook activo, se pausa y restaura al finalizar.
# No envíes mensajes reales durante esta breve prueba; el túnel no se modifica.
import copy
import socket
import types
import uuid
import threading
import time
import requests
import uvicorn
from fastapi import FastAPI

def diagnosticar_challenge_local():
    global servidor, hilo_servidor
    nombres = (
        "normalizar_numero_mx", "generar_respuesta", "extraer_mensajes_texto",
        "procesar_respuesta", "recibir_webhook",
    )
    if any(nombre not in globals() for nombre in nombres) or "pedidos" not in globals():
        raise RuntimeError("Ejecuta primero las definiciones del Challenge.")
    # Evita probar por accidente la función del Hands-On que llama al modelo.
    if "pedidos" not in generar_respuesta.__code__.co_names:
        raise RuntimeError("La función activa no es la consulta por reglas del Challenge.")

    entorno = dict(globals())
    recibidos, capturados = [], []
    entorno.update(
        WHATSAPP_PHONE_NUMBER_ID="000000000000000",
        WHATSAPP_ACCESS_TOKEN="", WHATSAPP_VERIFY_TOKEN="", WHATSAPP_URL="",
        NGROK_AUTHTOKEN="", pedidos=copy.deepcopy(pedidos),
        mensajes_recibidos=recibidos, usuarios_conocidos=set(),
        mensajes_en_proceso=set(), mensajes_procesados=set(),
        bloqueos_usuarios={}, lock_registros=threading.Lock(),
    )
    def capturar_envio(numero, texto):
        capturados.append({"destinatario_ficticio": numero, "respuesta": texto})
        return True  # Aceptación SIMULADA; no se llama a Cloud API.
    entorno["enviar_respuesta"] = capturar_envio
    for nombre in nombres:
        funcion = globals()[nombre]
        entorno[nombre] = types.FunctionType(
            funcion.__code__, entorno, nombre, funcion.__defaults__, funcion.__closure__,
        )
        entorno[nombre].__annotations__ = dict(funcion.__annotations__)

    app_local = FastAPI()
    app_local.post("/webhook")(entorno["recibir_webhook"])
    servidor_local = uvicorn.Server(uvicorn.Config(
        app_local, host="127.0.0.1", port=8000, access_log=False, log_level="warning",
    ))
    hilo_local = threading.Thread(target=servidor_local.run, daemon=True)
    anterior = globals().get("servidor")
    hilo_anterior = globals().get("hilo_servidor")
    restaurar = bool(anterior and hilo_anterior and hilo_anterior.is_alive())
    resultados = []
    casos = [
        ("1001", "Pedido 1001: En preparación (dato de ejemplo)."),
        ("1002", "Pedido 1002: Enviado (dato de ejemplo)."),
        ("1003", "Pedido 1003: Entregado (dato de ejemplo)."),
        ("9999", "Pedido no encontrado en el catálogo de ejemplo. Verifica el número e inténtalo de nuevo."),
        ("Hola", "Entrada inválida. Envía solo el número de pedido, por ejemplo: 1001."),
        ("pedido 1001", "Entrada inválida. Envía solo el número de pedido, por ejemplo: 1001."),
        ("01001", "Pedido no encontrado en el catálogo de ejemplo. Verifica el número e inténtalo de nuevo."),
        ("１００１", "Entrada inválida. Envía solo el número de pedido, por ejemplo: 1001."),
    ]
    print("SIMULACIÓN LOCAL: HTTP real a 127.0.0.1; envío a WhatsApp sustituido por captura.")
    try:
        if restaurar:
            anterior.should_exit = True
            hilo_anterior.join(timeout=180)
            if hilo_anterior.is_alive():
                anterior.should_exit = False
                restaurar = False
                raise RuntimeError("El servidor anterior no terminó; prueba cancelada.")
        # No enviar datos si otro proceso ajeno ocupa el puerto.
        with socket.socket() as comprobacion:
            if comprobacion.connect_ex(("127.0.0.1", 8000)) == 0:
                raise RuntimeError("El puerto 8000 está ocupado por otro servidor; no se enviaron casos.")
        hilo_local.start()
        limite = time.monotonic() + 10
        while not servidor_local.started:
            if not hilo_local.is_alive() or time.monotonic() > limite:
                raise RuntimeError("No se pudo iniciar el servidor local de diagnóstico.")
            time.sleep(0.05)
        with requests.Session() as cliente:
            cliente.trust_env = False  # No usar proxies para las peticiones locales.
            for indice, (entrada, esperado) in enumerate(casos):
                identificador = "wamid.LOCAL." + uuid.uuid4().hex
                payload = {
                    "object": "whatsapp_business_account",
                    "entry": [{"id": "000000000000001", "changes": [{
                        "field": "messages", "value": {
                            "messaging_product": "whatsapp",
                            "metadata": {"display_phone_number": "0000000000",
                                         "phone_number_id": "000000000000000"},
                            "contacts": [{"profile": {"name": "Usuario ficticio"},
                                          "wa_id": "5215500000000"}],
                            "messages": [{"from": "5215500000000", "id": identificador,
                                          "timestamp": "1700000000", "type": "text",
                                          "text": {"body": entrada}}],
                        },
                    }]}],
                }
                respuesta_http = cliente.post(
                    "http://127.0.0.1:8000/webhook", json=payload, timeout=10,
                )
                # ACK y tarea en segundo plano no terminan necesariamente juntos.
                limite = time.monotonic() + 10
                while identificador not in entorno["mensajes_procesados"]:
                    if time.monotonic() > limite:
                        raise RuntimeError(f"No terminó la tarea local para {entrada!r}.")
                    time.sleep(0.01)
                recibido = next(
                    dato for dato in recibidos
                    if dato["entry"][0]["changes"][0]["value"]["messages"][0]["id"] == identificador
                )
                texto_capturado = capturados[indice]["respuesta"]
                respuesta_esperada = (SALUDO_INICIAL + "\n" if indice == 0 else "") + esperado
                correcto = (
                    recibido == payload and respuesta_http.status_code == 200
                    and respuesta_http.json() == {"status": "ok"}
                    and texto_capturado == respuesta_esperada
                )
                registro = {"entrada": entrada, "payload_recibido": copy.deepcopy(recibido),
                            "http_status": respuesta_http.status_code, "ack": respuesta_http.json(),
                            "respuesta_capturada": texto_capturado, "correcto": correcto}
                resultados.append(registro)
                print(json.dumps(registro, ensure_ascii=False, indent=2))
        assert all(r["correcto"] for r in resultados), "Hay diferencias con las reglas esperadas."
        print(f"{len(resultados)}/{len(casos)} casos locales correctos. NO es evidencia E2E de WhatsApp.")
        return resultados
    finally:
        servidor_local.should_exit = True
        if hilo_local.ident is not None:
            hilo_local.join(timeout=15)
        if hilo_local.is_alive():
            raise RuntimeError("El servidor local sigue activo; no se restauró el servidor anterior.")
        if restaurar:
            servidor = uvicorn.Server(anterior.config)
            hilo_servidor = threading.Thread(target=servidor.run, daemon=True)
            hilo_servidor.start()
            limite = time.monotonic() + 10
            while not servidor.started:
                if not hilo_servidor.is_alive() or time.monotonic() > limite:
                    raise RuntimeError("No se pudo restaurar el servidor anterior.")
                time.sleep(0.05)

resultados_diagnostico_local = diagnosticar_challenge_local()


SIMULACIÓN LOCAL: HTTP real a 127.0.0.1; envío a WhatsApp sustituido por captura.
{
  "entrada": "1001",
  "payload_recibido": {
    "object": "whatsapp_business_account",
    "entry": [
      {
        "id": "000000000000001",
        "changes": [
          {
            "field": "messages",
            "value": {
              "messaging_product": "whatsapp",
              "metadata": {
                "display_phone_number": "0000000000",
                "phone_number_id": "000000000000000"
              },
              "contacts": [
                {
                  "profile": {
                    "name": "Usuario ficticio"
                  },
                  "wa_id": "5215500000000"
                }
              ],
              "messages": [
                {
                  "from": "5215500000000",
                  "id": "wamid.LOCAL.1850e42a6d0544d09d549af62b538196",
                  "timestamp": "1700000000",
                  "type": "text",
                  "te